In [1]:
from scipy.optimize import linprog

c = [-55, -65]

A_ub = [
    [4, 3],
    [3, 5],
    [1, 2],
]

b_ub = [320, 260, 110]

bounds = [(0, None), (0, None)]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print("Статус виконання:", res.status)
print("Повідомлення:", res.message)
print("Оптимальні значення (x1, x2):", res.x)
print("Мінімізоване значення (-Z):", res.fun)
print("Максимальний прибуток (Z):", -res.fun)

Статус виконання: 0
Повідомлення: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Оптимальні значення (x1, x2): [74.54545455  7.27272727]
Мінімізоване значення (-Z): -4572.727272727273
Максимальний прибуток (Z): 4572.727272727273


In [2]:
slack_values = b_ub - (A_ub @ res.x)

resources = ["Сировина (кг)", "Час обладнання (год)", "Електроенергія (кВт·год)"]

for name, s in zip(resources, slack_values):
  print(f"{name}: запас = {s:.2f} од. (активне: {abs(s) < 1e-5})")

Сировина (кг): запас = 0.00 од. (активне: True)
Час обладнання (год): запас = 0.00 од. (активне: True)
Електроенергія (кВт·год): запас = 20.91 од. (активне: False)


In [3]:
import numpy as np

A_ub_mod1 = [[4, 3], [3, 5], [1, 2]]
b_ub_mod1 = [320 * 1.15, 260, 110]  # сировина +15%

res_mod1 = linprog(
    c, A_ub=A_ub_mod1, b_ub=b_ub_mod1, bounds=bounds, method="highs"
)
print("--- Збільшення ліміту активного ресурсу на 15% ---")
print(f"Новий прибуток: {-res_mod1.fun:.2f} грн (зріс порівняно з {-res.fun:.2f})")

b_ub_mod2 = [320, 260, 110 * 1.5]
res_mod2 = linprog(
    c, A_ub=A_ub, b_ub=b_ub_mod2, bounds=bounds, method="highs"
)
print("\n--- Збільшення ліміту неактивного ресурсу на 50% ---")
print(
    f"Новий прибуток: {-res_mod2.fun:.2f} грн (змінився: {-res_mod2.fun == -res.fun})"
)

--- Збільшення ліміту активного ресурсу на 15% ---
Новий прибуток: 4766.67 грн (зріс порівняно з 4572.73)

--- Збільшення ліміту неактивного ресурсу на 50% ---
Новий прибуток: 4572.73 грн (змінився: True)


### Відповіді на контрольні питання

#### 1. Чому `scipy.optimize.linprog` завжди мінімізує цільову функцію і що потрібно для максимізації?

* **Чому мінімізує:** Це стандартна математична угода для більшості алгоритмів лінійного програмування (зокрема симплекс-методу та методів внутрішньої точки на кшталт HiGHS). Будь-яку задачу максимізації можна звести до мінімізації, оскільки знаходження найбільшого значення функції $Z(x)$ еквівалентне знаходженню найменшого значення оберненої функції $-Z(x)$.
* **Що зробити для максимізації:** Усі коефіцієнти цільової функції у векторі `c` потрібно помножити на `-1`. Після отримання результату оптимальне значення прибутку слід взяти з протилежним знаком (тобто `Z = -res.fun`), тоді як самі змінні `res.x` залишаються без змін.

#### 2. Чому обмеження виду «не менше» ($\ge$) потрібно домножити на $-1$ перед передачею в `A_ub`, `b_ub`?

* Функція `linprog` приймає обмеження-нерівності виключно у вигляді верхньої межі: $A_{\text{ub}} \cdot x \le b_{\text{ub}}$.
* Якщо ваше ресурсне або технологічне обмеження задане у вигляді нерівності «не менше» (наприклад, $150x_1 + 100x_2 \ge 900$), воно не відповідає цьому стандарту. Якщо помножити обиומ частини нерівності на $-1$, знак змінюється на протилежний: $-150x_1 - 100x_2 \le -900$, що тепер повністю задовольняє вимозі синтаксису `linprog` для матриць `A_ub` та `b_ub`.

#### 3. У чому різниця між активним і неактивним обмеженням і чому активне — це «вузьке місце»?

* **Активне обмеження** — це ресурс, який у процесі виробництва використовується **повністю** (значення залишку `slack` дорівнює нулю або наближене до нуля). Це означає, що підприємство вичерпало його ліміт до останньої одиниці.
* **Неактивне обмеження** — це ресурс, який залишається з певним **запасом** (`slack > 0`).
* **Чому це «вузьке місце»:** Саме активні обмеження стримують подальше зростання прибутку. Якщо підприємство захоче випустити більше продукції, йому не вистачить саме активного ресурсу. Тому інвестиції чи закупівлі мають сенс лише для розширення активних ресурсів — розширення неактивних нічого не змінить, бо вони й так споживаються не повністю.

#### 4. Чи узгоджується зміна прибутку в Завданні 5 з тим, яке обмеження збільшували?

* **Так, повністю узгоджується.**
* Коли ми збільшили ліміт **активного** ресурсу (сировини) на 15%, максимальний тижневий прибуток зріс, оскільки зникло головне стримуюче «вузьке місце», дозволивши перерозподілити план випуску.
* Коли ми збільшили ліміт **неактивного** ресурсу (електроенергії) на 50%, прибуток **не змінився взагалі**, оскільки цього ресурсу і до того було вдосталь (існував вільний запас), і він не обмежував виробництво. Це наочно підтверджує теорію лінійного програмування.